In [ ]:
!pip install datasets==3.6.0 huggingface_hub==0.34.4

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
import re
from collections import Counter
from torch.nn.utils.rnn import pack_padded_sequence
device='cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [ ]:
dataset=load_dataset("imdb")

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [ ]:
dataset["train"][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
for i in range(3):
  print("="*80)
  print("Lable:",dataset["train"][i]["label"])
  print()
  print(dataset["train"][i]["text"][:500])
  print()

Lable: 0

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attent

Lable: 0

"I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn't true. I've seen R-rated films with male nudity. Granted, they only offer some fleeting views, but where are the R-rated films with gaping vulvas and flapping labia? Nowhere, because they don't exist. The same goes for those 

In [ ]:
def tokenize(text):
  text=text.lower()
  text=re.sub(r"[^a-z0-9\s]","",text)
  tokens=text.split()

  return tokens

In [ ]:
sample = "I absolutely LOVED this movie!!! It was 10/10."

print(tokenize(sample))

['i', 'absolutely', 'loved', 'this', 'movie', 'it', 'was', '1010']


In [ ]:
review=dataset["train"][0]["text"]

tokens=tokenize(review)

tokens[:50]

['i',
 'rented',
 'i',
 'am',
 'curiousyellow',
 'from',
 'my',
 'video',
 'store',
 'because',
 'of',
 'all',
 'the',
 'controversy',
 'that',
 'surrounded',
 'it',
 'when',
 'it',
 'was',
 'first',
 'released',
 'in',
 '1967',
 'i',
 'also',
 'heard',
 'that',
 'at',
 'first',
 'it',
 'was',
 'seized',
 'by',
 'us',
 'customs',
 'if',
 'it',
 'ever',
 'tried',
 'to',
 'enter',
 'this',
 'country',
 'therefore',
 'being',
 'a',
 'fan',
 'of',
 'films']

In [ ]:
word_counter=Counter()

for review in dataset["train"]["text"]:
  tokens=tokenize(review)
  word_counter.update(tokens)

In [ ]:
print(word_counter.most_common(20))

[('the', 334713), ('and', 162228), ('a', 161946), ('of', 145326), ('to', 135042), ('is', 106855), ('in', 93028), ('it', 77101), ('i', 75719), ('this', 75190), ('that', 69352), ('br', 57143), ('was', 48008), ('as', 46662), ('for', 43964), ('with', 43871), ('movie', 41808), ('but', 41741), ('film', 37456), ('on', 33506)]


In [ ]:
MIN_FREQ=2

In [ ]:
word2idx={
    "<PAD>":0,
    "<UNK>":1
}

In [ ]:
for word, freq in word_counter.items():
    if freq >= MIN_FREQ:
        word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}

In [ ]:
print("Vocabulary size:", len(word2idx))

Vocabulary size: 57469


In [ ]:
for i, (word, idx) in enumerate(word2idx.items()):
    print(word, "->", idx)

    if i == 20:
        break

<PAD> -> 0
<UNK> -> 1
i -> 2
rented -> 3
am -> 4
curiousyellow -> 5
from -> 6
my -> 7
video -> 8
store -> 9
because -> 10
of -> 11
all -> 12
the -> 13
controversy -> 14
that -> 15
surrounded -> 16
it -> 17
when -> 18
was -> 19
first -> 20


In [ ]:
print("movie" in word2idx)
print("asdfghjk" in word2idx)

True
False


In [ ]:
def encode(tokens):
  return[
      word2idx.get(word,word2idx["<UNK>"])
      for word in tokens
  ]

In [ ]:
tokens = tokenize("I absolutely loved this movie")

print(tokens)

print(encode(tokens))

['i', 'absolutely', 'loved', 'this', 'movie']
[2, 577, 1729, 36, 377]


In [ ]:
tokens = tokenize("I love qwertyuiop")

print(tokens)
print(encode(tokens))

['i', 'love', 'qwertyuiop']
[2, 600, 1]


In [ ]:
MAX_LEN=200

In [ ]:
PAD_IDX=word2idx["<PAD>"]

In [ ]:
def pad_sequence(sequence):
  if len(sequence)<MAX_LEN:
    sequence=sequence+[PAD_IDX]*(MAX_LEN-len(sequence))
  else:
    sequence=sequence[:MAX_LEN]

  return sequence

In [ ]:
sequence = [10,20,30]

print(pad_sequence(sequence))

[10, 20, 30, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
print(len(pad_sequence(sequence)))

200


In [ ]:
review=dataset["train"][0]["text"]

tokens=tokenize(review)

encoded=encode(tokens)

padded=pad_sequence(encoded)

print("Tokens:", len(tokens))
print("Encoded:", len(encoded))
print("Padded:", len(padded))

Tokens: 288
Encoded: 288
Padded: 200


In [ ]:
class IMDBDataset(torch.utils.data.Dataset):
  def __init__(self,dataset):
    self.dataset=dataset

  def __len__(self):
    return len(self.dataset)
  def __getitem__(self,idx):
    review=self.dataset[idx]["text"]
    label=self.dataset[idx]["label"]

    tokens=tokenize(review)

    encoded=encode(tokens)

    padded=pad_sequence(encoded)
    length = min(len(encoded), MAX_LEN)
    return(
        torch.tensor(padded,dtype=torch.long),
        torch.tensor(label,dtype=torch.long),
        torch.tensor(length, dtype=torch.long)
    )


In [ ]:
train_data=IMDBDataset(dataset["train"])
test_data=IMDBDataset(dataset["test"])

In [ ]:
review,label,length=train_data[0]
print(review.shape)
print(label)
print(length.shape)

torch.Size([200])
tensor(0)
torch.Size([])


In [ ]:
BATCH_SIZE=32

train_loader=DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True
)
test_loader=DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
reviews, labels, length = next(iter(train_loader))

print(reviews.shape)
print(labels.shape)
print(length.shape)

print(length[:10])

torch.Size([32, 200])
torch.Size([32])
torch.Size([32])
tensor([ 83,  38, 183, 200, 200, 101, 145, 200,  36, 177])


In [ ]:
VOCAB_SIZE = len(word2idx)
EMBED_DIM = 64

embedding = nn.Embedding(
    num_embeddings=VOCAB_SIZE,
    embedding_dim=EMBED_DIM,
    padding_idx=PAD_IDX
)

In [ ]:
reviews, labels,length = next(iter(train_loader))

print(reviews.shape)

torch.Size([32, 200])


In [ ]:
embedded = embedding(reviews)

print(embedded.shape)

torch.Size([32, 200, 64])


In [ ]:
print(reviews[0, 0])          # First word ID
print(embedded[0, 0])         # Corresponding vector
print(embedded[0, 0].shape)

tensor(1375)
tensor([-1.4361,  0.6798, -0.4271, -1.1966,  1.0296,  0.1295,  0.1814, -0.6085,
         0.6177, -0.1971, -1.1420,  0.0335,  1.2444, -0.8277,  0.0731, -0.1432,
        -0.9734, -0.2587, -0.7733, -2.0738,  0.6137, -0.3666, -1.9513,  0.4394,
        -0.7514,  0.1245,  0.1089, -0.5011, -0.1053,  0.1583, -0.3475,  0.1015,
        -0.0100, -0.4619,  0.2181, -2.0017,  0.2730,  0.4354,  0.8265, -0.7635,
        -0.8765,  0.2474, -0.7613,  0.4647, -0.6588, -0.3697, -0.2795, -0.1015,
         0.3035, -0.3202,  2.4639,  1.5124, -0.3057,  1.1264,  1.2183,  0.2928,
        -0.8653,  1.1207,  0.0570, -0.2233,  0.7032,  0.0966, -0.7889, -0.9622],
       grad_fn=<SelectBackward0>)
torch.Size([64])


In [ ]:
class SentimentRNN(nn.Module):
  def __init__(self,vocab_size,embed_dim,hidden_dim):
    super().__init__()

    self.embedding=nn.Embedding(
        vocab_size,
        embed_dim,
        padding_idx=PAD_IDX
    )

    self.rnn=nn.LSTM(
        input_size=embed_dim,
        hidden_size=hidden_dim,
        batch_first=True
    )

    self.fc=nn.Linear(
        hidden_dim,
        2
    )

  def forward(self,x,length):
    embedded=self.embedding(x)
    packed = pack_padded_sequence(
            embedded,
            length.cpu(),
            batch_first=True,
            enforce_sorted=False
        )
    _,(hidden,cell)=self.rnn(packed)
    hidden=hidden.squeeze(0)
    logits=self.fc(hidden)

    return logits

In [ ]:
HIDDEN_DIM = 128

model = SentimentRNN(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM
)

print(model)

SentimentRNN(
  (embedding): Embedding(57469, 64, padding_idx=0)
  (rnn): LSTM(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


In [ ]:
reviews, labels,length = next(iter(train_loader))
model.to('cpu')
logits = model(reviews,length)
model = model.to(device)
print(logits.shape)

torch.Size([32, 2])


In [ ]:
def train_one_epoch(model,dataloader,criterion,optimizer):
  model.train()
  total_loss=0
  for i, (reviews, labels,length) in enumerate(dataloader):
        reviews = reviews.to(device)
        labels = labels.to(device)
        length=length.to(device)
        if i % 100 == 0:
            print(f"Batch {i}/{len(dataloader)}")
        optimizer.zero_grad()
        logits=model(reviews,length)
        loss=criterion(logits,labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss+=loss.item()

  return total_loss/len(dataloader)

In [ ]:
EPOCHS=5
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
for epoch in range(EPOCHS):
  loss=train_one_epoch(
      model,
      train_loader,
      criterion,
      optimizer
  )
  print(f"Epoch {epoch+1}: Loss = {loss:.4f}")

Batch 0/782
Batch 100/782
Batch 200/782
Batch 300/782
Batch 400/782
Batch 500/782
Batch 600/782
Batch 700/782
Epoch 1: Loss = 0.6650
Batch 0/782
Batch 100/782
Batch 200/782
Batch 300/782
Batch 400/782
Batch 500/782
Batch 600/782
Batch 700/782
Epoch 2: Loss = 0.5643
Batch 0/782
Batch 100/782
Batch 200/782
Batch 300/782
Batch 400/782
Batch 500/782
Batch 600/782
Batch 700/782
Epoch 3: Loss = 0.4528
Batch 0/782
Batch 100/782
Batch 200/782
Batch 300/782
Batch 400/782
Batch 500/782
Batch 600/782
Batch 700/782
Epoch 4: Loss = 0.3440
Batch 0/782
Batch 100/782
Batch 200/782
Batch 300/782
Batch 400/782
Batch 500/782
Batch 600/782
Batch 700/782
Epoch 5: Loss = 0.2669


In [ ]:
def evaluate(model,dataloader):
  model.eval()

  correct=0
  total=0

  with torch.no_grad():
    for reviews,labels,length in dataloader:
      reviews = reviews.to(device)
      labels = labels.to(device)
      length=length.to(device)
      logits=model(reviews,length)
      predictions=torch.argmax(logits,dim=1)
      correct+=(predictions==labels).sum().item()
      total+=labels.size(0)
  return correct/total

In [ ]:
accuracy = evaluate(model, test_loader)

print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.8260


In [ ]:
reviews, labels,length = next(iter(test_loader))

reviews = reviews.to(device)
labels = labels.to(device)

model.eval()

with torch.no_grad():
    logits = model(reviews,length)
    predictions = torch.argmax(logits, dim=1)

print("Predictions:", predictions[:20].cpu())
print("Labels      :", labels[:20].cpu())

Predictions: tensor([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0])
Labels      : tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix